# Behavioral Analysis: Agent Trajectories × Reference Patch Structure

**What:** Does the structural complexity of the reference (human) patch predict how agents behave, and does behavioral alignment with the reference procedure predict success?

**Data:**
- `output/trajectories/lite_all_models.parquet` — 589 runs on SWE-Bench Lite, 2 agents: SWE-Agent + GPT-4 (300 runs, 18% pass) and SWE-Agent + Claude 3.5 Sonnet (289 runs, 24% pass)
- `output/datasets/swe_bench_lite_resolved/test.parquet` — structural features of reference patches for the same 300 instances

**Data quality notes applied:**
- 142 runs (24%) hit token/cost limits (`exit_cost`) — artificially truncated. Filtered out of all process analyses.
- `edits_ops`, `edits_delta`, and `motifs_len` are redundant (Spearman ρ = 1.0). Only `edits_ops` and `modules_count` used as structural features.
- `hop_distance_min` is 97% null — excluded.

Plots precomputed by running the code cells below. Re-run to regenerate.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import altair as alt
import pandas as pd
from IPython.display import Image, display
from scipy.stats import spearmanr

ROOT = Path('../').resolve()
PLOTS_DIR = Path('plots/behavioral').resolve()
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    '20240402_sweagent_gpt4': 'SWE-Agent + GPT-4',
    '20240620_sweagent_claude3.5sonnet': 'SWE-Agent + Claude 3.5',
}
## Wong colorblind-safe: GPT-4 = blue, Claude = amber
MODEL_COLORS = ['#0072B2', '#E69F00']

def show(filename: str, width: int = 680):
    p = PLOTS_DIR / filename
    display(Image(str(p), width=width)) if p.exists() else print(f'Missing: {filename}')

In [ ]:
traj_raw = pd.read_parquet(ROOT / 'output/trajectories/lite_all_models.parquet')
ref_raw = pd.read_parquet(ROOT / 'output/datasets/swe_bench_lite_resolved/test.parquet')
import numpy as np

def structural_features(row) -> dict:
    def parse(x): return json.loads(x) if isinstance(x, str) else (x or [])
    edits = parse(row.get('edits'))
    modules = parse(row.get('modules'))
    return {
        'edits_ops': sum(len(c.get('operations', [])) for c in edits if isinstance(c, dict)),
        'modules_count': len(modules) if isinstance(modules, list) else 0,
    }

ref_feats = pd.concat(
    [ref_raw[['instance_id']], ref_raw.apply(structural_features, axis=1, result_type='expand')],
    axis=1,
)

traj_raw['model_label'] = traj_raw['model_id'].map(MODELS)
traj_raw['exit_cost'] = traj_raw['exit_status'].str.contains('exit_cost', na=False)
traj = traj_raw[~traj_raw['exit_cost']].copy()  # 447 organic runs
df = traj.merge(ref_feats, on='instance_id', how='inner')

print(f'All runs: {len(traj_raw)} | Organic (no exit-cost): {len(traj)} | Joined: {len(df)}')
print(f'Exit-cost filtered: {traj_raw["exit_cost"].sum()} ({100*traj_raw["exit_cost"].mean():.0f}%)')
print()
print(df.groupby('model_label')[['n_steps','n_edits','edit_retry_rate','passed']].mean().round(3))

## Reference patch complexity: solved vs unsolved tasks

In [ ]:
df_all = traj_raw.merge(ref_feats, on='instance_id', how='inner')
ref_by_outcome = df_all.drop_duplicates('instance_id')[['instance_id','edits_ops','modules_count']]
any_pass = df_all.groupby('instance_id')['passed'].any().reset_index()
ref_by_outcome = ref_by_outcome.merge(any_pass, on='instance_id')
ref_by_outcome['outcome'] = ref_by_outcome['passed'].map({True: 'solved by any agent', False: 'unsolved'})

print('Mean reference patch complexity by outcome:')
print(ref_by_outcome.groupby('outcome')[['edits_ops','modules_count']].mean().round(1))

for feat, title in [('edits_ops','Edit operations in reference patch'), ('modules_count','Files changed')]:
    (
        alt.Chart(ref_by_outcome).mark_bar(opacity=0.7).encode(
            alt.X(f'{feat}:Q', bin=alt.Bin(maxbins=25), title=title),
            alt.Y('count()', title=''),
            alt.Color('outcome:N', scale=alt.Scale(
                domain=['solved by any agent','unsolved'], range=['#0072B2','#E69F00'])),
        ).properties(width=320, height=180, title=title)
    ).save(PLOTS_DIR / f'patch_complexity_{feat}.png')

In [ ]:
show('patch_complexity_edits_ops.png')
show('patch_complexity_modules_count.png')

Solved tasks concentrate in the low edit-ops range. Unsolved tasks dominate the tail. Mean edits_ops: **1871 solved vs 3222 unsolved** (~72% higher for unsolved). This is task difficulty, not process — the reference patch is the target, not what the agent did.

## Behavioral-structural correlation

Does reference patch complexity predict how the agent behaves? Exit-cost runs excluded. Black = p < 0.05.

In [ ]:
struct_feats = ['edits_ops', 'modules_count']
beh_feats = ['n_steps', 'n_edits', 'n_searches', 'n_runs', 'edit_retry_rate']

rows = []
for model_id, model_label in MODELS.items():
    sub = df[df['model_id'] == model_id]
    for b in beh_feats:
        for s in struct_feats:
            rho, p = spearmanr(sub[b], sub[s])
            rows.append({'model': model_label, 'behavioral': b, 'structural': s,
                         'rho': round(rho, 3), 'rho_str': f'{rho:+.2f}', 'significant': p < 0.05})

corr_df = pd.DataFrame(rows)

charts = []
for model_label in MODELS.values():
    sub = corr_df[corr_df['model'] == model_label]
    hm = alt.Chart(sub).mark_rect().encode(
        alt.X('structural:N', title=''), alt.Y('behavioral:N', title=''),
        alt.Color('rho:Q', scale=alt.Scale(scheme='redblue', domainMid=0, domain=[-0.4, 0.4]), title='ρ'),
    )
    tx = alt.Chart(sub).mark_text(fontSize=11).encode(
        alt.X('structural:N'), alt.Y('behavioral:N'), alt.Text('rho_str:N'),
        color=alt.condition('datum.significant', alt.value('black'), alt.value('#aaa')),
    )
    charts.append((hm + tx).properties(width=150, height=180, title=model_label))

alt.hconcat(*charts).save(PLOTS_DIR / 'behavioral_structural_corr.png')

In [ ]:
show('behavioral_structural_corr.png', width=500)

`n_searches` is the only significant signal (ρ ≈ +0.16–0.20): agents search more when the reference patch touches more modules. Everything else — steps, edits, retry rate — is near-zero. Reference patch complexity does not drive how much the agent works.

## Behavioral stats by outcome

In [ ]:
print(df.groupby(['model_label','passed'])[['n_steps','n_edits','edit_retry_rate']].mean().round(2))

df['outcome'] = df['passed'].map({True: 'solved', False: 'failed'})
beh_charts = []
for col, title in [('n_steps','Steps taken'), ('n_edits','Edit actions'), ('edit_retry_rate','Edit retry rate')]:
    beh_charts.append(
        alt.Chart(df).mark_boxplot(extent=1.5).encode(
            alt.X('model_label:N', title=''), alt.Y(f'{col}:Q', title=title),
            alt.Color('outcome:N', scale=alt.Scale(domain=['solved','failed'], range=['#0072B2','#D55E00'])),
        ).properties(width=200, height=200, title=title)
    )
alt.hconcat(*beh_charts).save(PLOTS_DIR / 'behavioral_by_outcome.png')

In [ ]:
show('behavioral_by_outcome.png')

## Sequence alignment: does matching the passing procedure predict success?

In [ ]:
def levenshtein(a: list, b: list) -> int:
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, n + 1):
            dp[j] = prev[j-1] if a[i-1] == b[j-1] else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]

align_rows = []
for model_id, model_label in MODELS.items():
    sub = df[df['model_id'] == model_id].copy()
    passed_seqs = [s.split() for s in sub[sub['passed']]['action_sequence']]
    template = Counter([' '.join(s) for s in passed_seqs]).most_common(1)[0][0].split()
    print(f'{model_label} template: {" → ".join(template[:8])}{" …" if len(template) > 8 else ""}')
    sub['align_dist'] = sub['action_sequence'].apply(lambda s: levenshtein(s.split(), template))
    sub['align_bucket'] = pd.qcut(sub['align_dist'], q=5,
        labels=['Q1 closest','Q2','Q3','Q4','Q5 farthest'], duplicates='drop')
    agg = sub.groupby('align_bucket', observed=True).agg(
        n=('passed','count'), pass_rate=('passed','mean')).reset_index()
    agg['model'] = model_label
    align_rows.append(agg)

align_df = pd.concat(align_rows, ignore_index=True)
print()
print(align_df.to_string(index=False))

charts = []
for model_label in MODELS.values():
    sub = align_df[align_df['model'] == model_label]
    charts.append(
        alt.Chart(sub).mark_bar().encode(
            alt.X('align_bucket:N', sort=None, title='distance to passing procedure template'),
            alt.Y('pass_rate:Q', title='pass rate', scale=alt.Scale(domain=[0, 0.55])),
            alt.Color('model:N', scale=alt.Scale(domain=list(MODELS.values()), range=MODEL_COLORS), legend=None),
        ).properties(width=220, height=200, title=model_label)
    )
alt.hconcat(*charts).properties(title='Pass rate by alignment to passing procedure').save(
    PLOTS_DIR / 'sequence_alignment.png')

In [ ]:
show('sequence_alignment.png')

Claude 3.5 shows a clean monotonic gradient: Q1 (closest to the passing template) has ~43% pass rate, dropping to ~13% at Q5. GPT-4 is flatter and noisier — procedural alignment matters more for Claude, suggesting they use structurally different implicit strategies.

---
## Summary

| Finding | Detail |
|---|---|
| Solved tasks have simpler reference patches | Mean edits_ops: 1871 (solved) vs 3222 (unsolved) — task difficulty signal, not process |
| Reference complexity doesn't drive agent process | Correlations near-zero except n_searches (ρ ≈ +0.18) |
| Procedural alignment predicts success | Q1 closest to passing template: 2–3× higher pass rate than Q5, especially for Claude |
| Exit-cost truncation is a real confound | 24% of runs hit token limits — filtered for all process analyses |

**Next:** Extract structural representations from agent edit sequences (not just action-type strings) to compare agent edits structurally against the reference patch — the proper alignment measure.